# ApexPlanet Internship — Task 4: Statistics, Clustering & Predictive Modeling

**Dataset:** UCI Online Retail  
**Environment:** Google Colab

All numerical findings are calculated from the uploaded dataset. Do not invent p-values, cluster counts, accuracy, or business conclusions.

In [ ]:
!pip -q install pandas openpyxl scipy scikit-learn statsmodels matplotlib seaborn

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

file_name = next(iter(uploaded))
df = pd.read_excel(file_name)
print(df.shape)
display(df.head())

## 1. Cleaning and feature engineering

In [ ]:
clean = df.copy()
clean = clean.drop_duplicates()
clean["InvoiceNo"] = clean["InvoiceNo"].astype(str)
clean = clean[~clean["InvoiceNo"].str.startswith("C")]
clean = clean[clean["Quantity"] > 0]
clean = clean[clean["UnitPrice"] > 0]
clean = clean.dropna(subset=["CustomerID"])
clean["InvoiceDate"] = pd.to_datetime(clean["InvoiceDate"])
clean["CustomerID"] = clean["CustomerID"].astype(int)
clean["TotalSales"] = clean["Quantity"] * clean["UnitPrice"]
clean["Year"] = clean["InvoiceDate"].dt.year
clean["Month"] = clean["InvoiceDate"].dt.to_period("M").astype(str)

print("Cleaned shape:", clean.shape)

## 2. Descriptive statistics

In [ ]:
numeric_cols = ["Quantity", "UnitPrice", "TotalSales"]
stats_table = pd.DataFrame({
    "Mean": clean[numeric_cols].mean(),
    "Median": clean[numeric_cols].median(),
    "Mode": [clean[c].mode().iloc[0] for c in numeric_cols],
    "Std Dev": clean[numeric_cols].std(),
    "Skewness": clean[numeric_cols].skew()
})
display(stats_table.round(2))
display(clean[numeric_cols].describe().T.round(2))

## 3. Hypothesis testing — t-test

Example: compare log-transformed transaction values between the United Kingdom and Germany using Welch's independent-samples t-test.

In [ ]:
uk = clean.loc[clean["Country"] == "United Kingdom", "TotalSales"]
germany = clean.loc[clean["Country"] == "Germany", "TotalSales"]

t_stat, p_value = stats.ttest_ind(
    np.log1p(uk),
    np.log1p(germany),
    equal_var=False
)
print("t-statistic:", round(t_stat,4))
print("p-value:", p_value)
print("Significant at 5%:", p_value < 0.05)

**H0:** mean log transaction value is equal between the groups.  
**H1:** mean log transaction value differs.  
Use α = 0.05. Statistical significance does not establish causation.

## 4. Chi-square test

Test whether country and a high/low transaction-value category are associated.

In [ ]:
median_sales = clean["TotalSales"].median()
clean["HighValue"] = np.where(clean["TotalSales"] >= median_sales, "High", "Low")

counts = clean["Country"].value_counts()
valid = counts[counts >= 50].index
contingency = pd.crosstab(
    clean.loc[clean["Country"].isin(valid), "Country"],
    clean.loc[clean["Country"].isin(valid), "HighValue"]
)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print("Chi-square:", round(chi2,4))
print("p-value:", p_value)
print("Degrees of freedom:", dof)
print("Significant at 5%:", p_value < 0.05)

## 5. 95% confidence interval for mean transaction value

In [ ]:
x = clean["TotalSales"].dropna()
ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=len(x)-1,
    loc=x.mean(),
    scale=stats.sem(x)
)
print("Mean:", round(x.mean(),2))
print("95% CI:", (round(ci_low,2), round(ci_high,2)))

## 6. Time-series analysis

In [ ]:
monthly_sales = clean.set_index("InvoiceDate").resample("MS")["TotalSales"].sum()
display(monthly_sales.to_frame("Sales").round(2))

In [ ]:
plt.figure(figsize=(12,5))
monthly_sales.plot()
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.grid(True)
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

if len(monthly_sales) >= 24:
    decomposition = seasonal_decompose(monthly_sales, model="additive", period=12)
    decomposition.plot()
    plt.show()
else:
    print("Not enough observations for 12-month decomposition.")

In [ ]:
moving_average = monthly_sales.rolling(3).mean()
plt.figure(figsize=(12,5))
plt.plot(monthly_sales.index, monthly_sales.values, label="Actual")
plt.plot(moving_average.index, moving_average.values, label="3-Month Moving Average")
plt.title("Sales with 3-Month Moving Average")
plt.legend()
plt.grid(True)
plt.show()

## 7. Customer segmentation with RFM

In [ ]:
reference_date = clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = clean.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date-x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalSales", "sum")
).reset_index()

rfm_model = rfm.copy()
for col in ["Recency","Frequency","Monetary"]:
    rfm_model[col] = np.log1p(rfm_model[col])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_model[["Recency","Frequency","Monetary"]])

display(rfm.head())

## 8. Elbow method

In [ ]:
inertias=[]
K_range=range(2,9)
for k in K_range:
    km=KMeans(n_clusters=k,random_state=42,n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(list(K_range), inertias, marker="o")
plt.xlabel("K")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

Choose K from the elbow and business interpretability. The code below starts with 4 only as a placeholder; change it if the elbow suggests another value.

In [ ]:
optimal_k = 4  # Change after inspecting the elbow
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)

segment_profile = rfm.groupby("Cluster").agg(
    Customers=("CustomerID","count"),
    AvgRecency=("Recency","mean"),
    AvgFrequency=("Frequency","mean"),
    AvgMonetary=("Monetary","mean")
).round(2)

display(segment_profile)

## 9. PCA visualization

In [ ]:
pca=PCA(n_components=2,random_state=42)
X_pca=pca.fit_transform(X_scaled)
rfm["PC1"]=X_pca[:,0]
rfm["PC2"]=X_pca[:,1]

plt.figure(figsize=(9,6))
sns.scatterplot(data=rfm,x="PC1",y="PC2",hue="Cluster",palette="tab10",s=50)
plt.title("Customer Segments — PCA Projection")
plt.show()
print("Explained variance:", pca.explained_variance_ratio_)

## 10. Basic predictive model

We predict monthly sales using lagged sales features. Because this is time-series data, the train/test split is chronological rather than randomly shuffled.

In [ ]:
ts=monthly_sales.to_frame("Sales").reset_index()
ts.columns=["Date","Sales"]
ts["Lag1"]=ts["Sales"].shift(1)
ts["Lag2"]=ts["Sales"].shift(2)
ts["Lag3"]=ts["Sales"].shift(3)
ts["Rolling3"]=ts["Sales"].shift(1).rolling(3).mean()

model_data=ts.dropna()
X=model_data[["Lag1","Lag2","Lag3","Rolling3"]]
y=model_data["Sales"]

split=int(len(model_data)*0.8)
X_train,X_test=X.iloc[:split],X.iloc[split:]
y_train,y_test=y.iloc[:split],y.iloc[split:]

model=LinearRegression()
model.fit(X_train,y_train)
pred=model.predict(X_test)

print("R²:",round(r2_score(y_test,pred),4))
print("MAE:",round(mean_absolute_error(y_test,pred),2))
print("RMSE:",round(np.sqrt(mean_squared_error(y_test,pred)),2))

In [ ]:
importance=(pd.DataFrame({"Feature":X.columns,"Coefficient":model.coef_})
             .assign(AbsCoefficient=lambda d:d["Coefficient"].abs())
             .sort_values("AbsCoefficient",ascending=False))
display(importance)
print("Top 3 features:")
display(importance.head(3))

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(y_test.index,y_test.values,marker="o",label="Actual")
plt.plot(y_test.index,pred,marker="o",label="Predicted")
plt.title("Actual vs Predicted Monthly Sales")
plt.legend()
plt.grid(True)
plt.show()

## 11. Business findings template

Fill these with the actual outputs:

**Statistics**
- Mean transaction value: [value]
- Median: [value]
- Skewness: [value]
- t-test p-value: [value]
- Chi-square p-value: [value]
- 95% CI: [lower, upper]

**Segmentation**
- Highest monetary-value cluster: [cluster]
- Highest-frequency cluster: [cluster]
- Reactivation candidate cluster: [cluster]

**Prediction**
- R²: [value]
- MAE: [value]
- RMSE: [value]
- Top 3 features: [features]

**Recommendations**
1. Retain high-value/high-frequency customers.
2. Reactivate customers with high recency and weaker recent activity.
3. Use the baseline forecast as an input to sales planning, while monitoring model error.

## Task 4 checklist

- [x] Descriptive statistics
- [x] t-test
- [x] Chi-square
- [x] Confidence interval
- [x] Time-series resampling
- [x] Trend / decomposition / moving average
- [x] StandardScaler
- [x] K-Means / elbow
- [x] PCA
- [x] Segment profiling
- [x] Regression model
- [x] R² / MAE / RMSE
- [x] Top 3 model features
- [ ] 2-page final PDF
- [ ] Automation pipeline